# Official ColBERTv2 baseline on BEIR (Colab)

Runs the ORIGINAL ColBERTv2 pipeline with the SAME settings as the mini pipeline
(`query_maxlen=32`, `doc_maxlen=180`, `nbits=2`, top-100) so that
`Official` vs `Mini Base` vs `CAD` is a fair comparison on each dataset.

Results are written to `MyDrive/colbert-run/<dataset>/official_baseline.json`
(tiny, a few KB on Drive). `Mini_ColBERTv2_MultiBEIR.ipynb` reads it automatically.

**Order:** you can run this before or after the mini notebook; only the json on
Drive matters. Requires a Colab GPU (T4).

> Note: the numbers in the ColBERTv2 paper use `doc_maxlen=300`, so they are NOT
> directly comparable to the mini pipeline. This notebook re-runs official with
> `180` for a like-for-like comparison.

In [ ]:
# 1. Clone the official ColBERT repo (GPU required for indexing)
!nvidia-smi
!git clone https://github.com/stanford-futuredata/ColBERT.git || true
%cd ColBERT
!pip -q install -e .
!pip -q install faiss-gpu-cu12

In [ ]:
# 2. Pin transformers<5 (colbert-ai compatibility) + helpers
!pip install -q "transformers<5" jedi ninja
import torch, transformers
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("transformers", transformers.__version__)

In [ ]:
# 3. Clone/update the mini repo (baseline scripts live there; run them in place)
%cd /content
!git clone https://github.com/HoangNguyen689/colbert-hoangnd.git || true
!git -C colbert-hoangnd fetch --all
!git -C colbert-hoangnd reset --hard origin/main
MINI_DIR = '/content/colbert-hoangnd'
OFFICIAL = f'{MINI_DIR}/baselines/official'
!ls {OFFICIAL}

In [ ]:
# 4. Mount Google Drive (shared storage for the tiny official_baseline.json)
import os
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = Path('/content/drive/MyDrive/colbert-run')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print('Drive dir:', DRIVE_DIR)

In [ ]:
# 5. Run official prepare -> index+search -> evaluate for each dataset
#    Settings pinned to match the mini pipeline (query/doc maxlen, nbits=2, k=100).
import subprocess, sys
os.chdir('/content/ColBERT')
DATASETS = ['nfcorpus', 'scidocs', 'fiqa', 'arguana']
OPTIONAL_DATASETS = ['trec-covid']   # heavy (~14GB, needs high-RAM); uncomment next line to include
# DATASETS += OPTIONAL_DATASETS

def run(cmd):
    print('$', ' '.join(cmd), flush=True)
    subprocess.run(cmd, check=True)   # stop as soon as one step fails

for ds in DATASETS:
    print(f'\n==================== {ds} ====================')
    dst = DRIVE_DIR / ds
    dst.mkdir(parents=True, exist_ok=True)
    run([sys.executable, f'{OFFICIAL}/prepare_beir.py', '--dataset', ds, '--out', '/content/ColBERT/data'])
    run([sys.executable, f'{OFFICIAL}/run_official_colbertv2.py', '--dataset', ds,
         '--data', f'/content/ColBERT/data/{ds}_colbert', '--nbits', '2', '--k', '100',
         '--doc-maxlen', '180', '--query-maxlen', '32'])
    run([sys.executable, f'{OFFICIAL}/evaluate_official.py', '--dataset', ds,
         '--json-out', str(dst / 'official_baseline.json')])

In [ ]:
# 6. Print per-dataset comparison tables (Official vs Mini Base vs IDF/Adaptive/CAD)
#    Official comes from Drive; mini rankings come from Drive if the mini notebook synced them.
import json
from pathlib import Path
import sys
sys.path.insert(0, '/content/colbert-hoangnd')
from evaluation.metrics import evaluate, load_json

VARIANTS = ['official', 'original', 'idf', 'adaptive', 'cad']
LABELS = {'official': 'Official', 'original': 'Mini Base', 'idf': 'IDF',
          'adaptive': 'Adaptive', 'cad': 'CAD'}
DATASETS = ['nfcorpus', 'scidocs', 'fiqa', 'arguana']
OPTIONAL_DATASETS = ['trec-covid']   # heavy (~14GB, needs high-RAM); uncomment next line to include
# DATASETS += OPTIONAL_DATASETS

print(f"{'Dataset':<14} {'Method':<10} {'nDCG@10':>8} {'MRR@10':>8} {'Recall@100':>11}")
print('-' * 55)
for ds in DATASETS:
    d = DRIVE_DIR / ds
    qrels_path = d / 'qrels_test.json'
    if not qrels_path.exists():
        print(f"{ds:<14} (no mini qrels on Drive - run the Mini notebook first)")
        continue
    qrels = load_json(qrels_path)
    metrics = {}
    off = d / 'official_baseline.json'
    if off.exists():
        metrics['official'] = json.loads(off.read_text())
    for name in ['original', 'idf', 'adaptive', 'cad']:
        p = d / f'rankings_{name}.json'
        if p.exists():
            metrics[name] = evaluate(load_json(p), qrels)
    for name in VARIANTS:
        m = metrics.get(name)
        if m:
            print(f"{ds:<14} {LABELS[name]:<10} {m['nDCG@10']:>8.4f} {m['MRR@10']:>8.4f} {m['Recall@100']:>11.4f}")
    print('-' * 55)